# Proyecto final · Predicción de partidos ATP
## Paso 5 · Modelos

Tres modelos de familias distintas, comparados con **la misma validación** y con el preprocesamiento **dentro del Pipeline**:

| Modelo | Familia | Por qué |
|---|---|---|
| Regresión logística | Lineal | Simple e interpretable: sus coeficientes dicen cuánto pesa cada variable |
| Random forest | Árboles promediados (bagging) | Capta relaciones no lineales e interacciones |
| XGBoost | Árboles en secuencia (boosting) | Suele ser el más preciso en datos tabulares |

**Puntos de comparación** (no son modelos): línea base (siempre gana A), regla del ranking y regla del Elo.

### ¿Cómo se parten los datos?
Los partidos ocurren en el tiempo, así que **no se puede mezclar al azar**: el modelo estaría aprendiendo del futuro para predecir el pasado.
- **Entrenamiento:** 2001–2022. Aquí se comparan los modelos con validación cruzada temporal (`TimeSeriesSplit`): cada partición entrena con los años anteriores y valida con los siguientes.
- **Prueba:** 2023–2026. No se toca hasta el paso 6.

In [1]:
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, cross_validate
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

SEMILLA = 42
CARPETA = os.environ.get("TENIS_DATOS", "datos")

# True = busca hiperparámetros (tarda varios minutos).
# False = usa los mejores ya guardados: para correr en vivo en la exposición.
BUSCAR_HIPERPARAMETROS = not os.path.exists(os.path.join(CARPETA, "mejores_parametros.json"))

datos = pd.read_csv(os.path.join(CARPETA, "tabla_modelo.csv.gz"), low_memory=False, parse_dates=["fecha"])
with open(os.path.join(CARPETA, "columnas_modelo.json")) as f:
    cols = json.load(f)
NUMERICAS, CATEGORICAS = cols["numericas"], cols["categoricas"]

datos = datos[datos["anio"] >= 2001].sort_values(["fecha", "match_id"]).reset_index(drop=True)
train = datos[datos["anio"] <= 2022].reset_index(drop=True)
test = datos[datos["anio"] >= 2023].reset_index(drop=True)
X_train, y_train = train[NUMERICAS + CATEGORICAS], train["gana_A"]

print(f"Entrenamiento 2001–2022: {len(train):,} partidos (gana_A = {y_train.mean():.3f})")
print(f"Prueba 2023–2026:        {len(test):,} partidos (gana_A = {test['gana_A'].mean():.3f})")
print(f"Columnas: {len(NUMERICAS)} numéricas + {len(CATEGORICAS)} categóricas")
print(f"Modo: {'búsqueda de hiperparámetros' if BUSCAR_HIPERPARAMETROS else 'parámetros guardados (rápido)'}")

Entrenamiento 2001–2022: 65,124 partidos (gana_A = 0.497)
Prueba 2023–2026:        10,329 partidos (gana_A = 0.490)
Columnas: 29 numéricas + 3 categóricas
Modo: búsqueda de hiperparámetros


### Validación cruzada temporal
5 particiones. Cada una entrena con todo lo anterior y valida con el bloque siguiente, como si el modelo se usara de verdad año tras año.

In [2]:
cv = TimeSeriesSplit(n_splits=5)
for k, (tr, va) in enumerate(cv.split(X_train), 1):
    print(f"Partición {k}: entrena {train['fecha'].iloc[tr[0]].year}–{train['fecha'].iloc[tr[-1]].year} "
          f"({len(tr):,})  →  valida {train['fecha'].iloc[va[0]].year}–{train['fecha'].iloc[va[-1]].year} ({len(va):,})")

Partición 1: entrena 2001–2004 (10,854)  →  valida 2004–2007 (10,854)
Partición 2: entrena 2001–2007 (21,708)  →  valida 2007–2011 (10,854)
Partición 3: entrena 2001–2011 (32,562)  →  valida 2011–2014 (10,854)
Partición 4: entrena 2001–2014 (43,416)  →  valida 2014–2018 (10,854)
Partición 5: entrena 2001–2018 (54,270)  →  valida 2018–2022 (10,854)


### Preprocesamiento dentro del Pipeline
- **Numéricas:** se imputan con la mediana **de la partición de entrenamiento** (las de saque tienen un 7 % de faltantes) y se agrega una marca de "faltaba". Luego se estandarizan, que la logística necesita y a los árboles no les afecta.
- **Categóricas** (superficie, nivel, ronda): one-hot. Si aparece una categoría nueva en validación, se ignora.

Como todo está dentro del Pipeline, en cada partición la mediana y la escala se calculan **solo con los datos de entrenamiento**: la validación nunca "se filtra" hacia el preprocesamiento.

In [4]:
def preprocesador(numericas):
    return ColumnTransformer([
        ("num", Pipeline([("imputar", SimpleImputer(strategy="median", add_indicator=True)),
                          ("escalar", StandardScaler())]), numericas),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS),
    ])

def pipeline(modelo, numericas=NUMERICAS):
    return Pipeline([("prep", preprocesador(numericas)), ("modelo", modelo)])

METRICAS = {"exactitud": "accuracy", "auc": "roc_auc", "logloss": "neg_log_loss"}

### Puntos de comparación en las mismas particiones
Las reglas no se entrenan, pero se evalúan en exactamente los mismos bloques de validación para que la comparación sea justa.

In [5]:
def evaluar_regla(nombre, pred):
    ex = [accuracy_score(y_train.iloc[va], pred[va]) for _, va in cv.split(X_train)]
    return {"Modelo": nombre, "exactitud": np.mean(ex), "exactitud_std": np.std(ex),
            "auc": np.nan, "logloss": np.nan, "segundos": 0}

pred_base = np.ones(len(train), dtype=int)
pred_rank = np.where(train["dif_log_rank"] < 0, 1, np.where(train["dif_log_rank"] > 0, 0, 1))
pred_elo = np.where(train["dif_elo"] >= 0, 1, 0)

resultados = [evaluar_regla("Línea base (siempre A)", pred_base),
              evaluar_regla("Regla: mejor ranking", pred_rank),
              evaluar_regla("Regla: mayor Elo", pred_elo)]
pd.DataFrame(resultados)[["Modelo", "exactitud"]].round(4)

,Modelo,exactitud
0,Línea base (siempre A),0.4974
1,Regla: mejor ranking,0.6639
2,Regla: mayor Elo,0.6800


### Modelo 1 · Regresión logística
En el paso 4 vimos dos pares de variables muy correlacionadas: ranking y puntos (−0,91), y Elo general y Elo por superficie (0,85). En un modelo lineal eso puede volver inestables los coeficientes. Antes de elegir, se prueba si quitar una de cada par cambia algo.

In [6]:
REDUNDANTES = ["dif_log_puntos", "dif_elo_sup"]
NUM_LOGISTICA_REDUCIDA = [c for c in NUMERICAS if c not in REDUNDANTES]

comparar = {}
for nombre, numericas in [("Todas las variables", NUMERICAS), ("Sin redundantes", NUM_LOGISTICA_REDUCIDA)]:
    r = cross_validate(pipeline(LogisticRegression(max_iter=3000), numericas), X_train, y_train, cv=cv, scoring=METRICAS)
    comparar[nombre] = {"exactitud": r["test_exactitud"].mean(), "auc": r["test_auc"].mean(), "logloss": -r["test_logloss"].mean()}
pd.DataFrame(comparar).T.round(4)

,exactitud,auc,logloss
Todas las variables,0.6944,0.7678,0.5726
Sin redundantes,0.6917,0.7621,0.5783


In [7]:
# Si la diferencia en AUC es menor a 0,001, se prefiere la versión más simple
dif_auc = comparar["Todas las variables"]["auc"] - comparar["Sin redundantes"]["auc"]
NUM_LOGISTICA = NUM_LOGISTICA_REDUCIDA if dif_auc < 0.001 else NUMERICAS
print(f"Diferencia de AUC: {dif_auc:+.4f} → la logística usa: "
      f"{'sin redundantes' if NUM_LOGISTICA is NUM_LOGISTICA_REDUCIDA else 'todas las variables'}")

Diferencia de AUC: +0.0057 → la logística usa: todas las variables


### Búsqueda de hiperparámetros
Una búsqueda pequeña para cada modelo, con la **misma validación temporal**. Se elige por log loss: premia las probabilidades bien calibradas, no solo acertar el ganador.

| Modelo | Qué se ajusta | Qué controla |
|---|---|---|
| Logística | `C` | Cuánta regularización (C pequeño = coeficientes más pequeños) |
| Random forest | `max_depth`, `min_samples_leaf` | Qué tan complejos son los árboles |
| XGBoost | `max_depth`, `learning_rate` | Complejidad de cada árbol y tamaño de cada paso |

In [8]:
MODELOS = {
    "Regresión logística": (
        pipeline(LogisticRegression(max_iter=3000), NUM_LOGISTICA),
        {"modelo__C": [0.01, 0.1, 1.0]}),
    "Random forest": (
        pipeline(RandomForestClassifier(n_estimators=300, max_features="sqrt", n_jobs=-1, random_state=SEMILLA)),
        {"modelo__max_depth": [8, 14], "modelo__min_samples_leaf": [20, 100]}),
    "XGBoost": (
        pipeline(XGBClassifier(n_estimators=400, subsample=0.8, colsample_bytree=0.8,
                               eval_metric="logloss", n_jobs=-1, random_state=SEMILLA)),
        {"modelo__max_depth": [3, 5], "modelo__learning_rate": [0.03, 0.08]}),
}

ruta_params = os.path.join(CARPETA, "mejores_parametros.json")
if BUSCAR_HIPERPARAMETROS:
    mejores = {}
    for nombre, (pipe, grilla) in MODELOS.items():
        inicio = time.time()
        busqueda = GridSearchCV(pipe, grilla, cv=cv, scoring="neg_log_loss", n_jobs=1)
        busqueda.fit(X_train, y_train)
        mejores[nombre] = busqueda.best_params_
        print(f"{nombre}: {busqueda.best_params_}  ({time.time() - inicio:.0f} s)")
    with open(ruta_params, "w") as f:
        json.dump(mejores, f, indent=1)
else:
    with open(ruta_params) as f:
        mejores = json.load(f)
    for nombre, p in mejores.items():
        print(f"{nombre}: {p}  (guardados)")

Regresión logística: {'modelo__C': 0.1}  (8 s)
Random forest: {'modelo__max_depth': 14, 'modelo__min_samples_leaf': 20}  (132 s)
XGBoost: {'modelo__learning_rate': 0.03, 'modelo__max_depth': 3}  (31 s)


### Comparación con la misma vara
Cada modelo, con sus mejores hiperparámetros, se evalúa en las mismas 5 particiones temporales.

In [9]:
finales = {}
por_particion = {}
for nombre, (pipe, _) in MODELOS.items():
    pipe.set_params(**mejores[nombre])
    inicio = time.time()
    r = cross_validate(pipe, X_train, y_train, cv=cv, scoring=METRICAS)
    resultados.append({"Modelo": nombre,
                       "exactitud": r["test_exactitud"].mean(), "exactitud_std": r["test_exactitud"].std(),
                       "auc": r["test_auc"].mean(), "logloss": -r["test_logloss"].mean(),
                       "segundos": time.time() - inicio})
    finales[nombre] = pipe
    por_particion[nombre] = r["test_exactitud"]
    print(f"{nombre}: exactitud {r['test_exactitud'].mean():.4f}  ({time.time() - inicio:.0f} s)")

tabla_cv = pd.DataFrame(resultados)
mostrar = tabla_cv.copy()
mostrar["Exactitud (%)"] = (mostrar["exactitud"] * 100).round(2).astype(str) + " ± " + (mostrar["exactitud_std"] * 100).round(2).astype(str)
mostrar["AUC"] = mostrar["auc"].round(4)
mostrar["Log loss"] = mostrar["logloss"].round(4)
mostrar["Tiempo (s)"] = mostrar["segundos"].round(0)
mostrar[["Modelo", "Exactitud (%)", "AUC", "Log loss", "Tiempo (s)"]]

Regresión logística: exactitud 0.6948  (3 s)
Random forest: exactitud 0.6955  (31 s)
XGBoost: exactitud 0.6949  (6 s)


,Modelo,Exactitud (%),AUC,Log loss,Tiempo (s)
0,Línea base (siempre A),49.74 ± 0.41,NaN,NaN,0.0
1,Regla: mejor ranking,66.39 ± 1.47,NaN,NaN,0.0
2,Regla: mayor Elo,68.0 ± 1.33,NaN,NaN,0.0
3,Regresión logística,69.48 ± 1.29,0.7679,0.5726,3.0
4,Random forest,69.55 ± 1.29,0.7666,0.5752,31.0
5,XGBoost,69.49 ± 1.31,0.7677,0.5726,6.0


In [10]:
# Exactitud en cada partición: ¿algún modelo gana siempre, o depende del periodo?
por_particion["Regla: mejor ranking"] = [accuracy_score(y_train.iloc[va], pred_rank[va]) for _, va in cv.split(X_train)]
por_particion["Regla: mayor Elo"] = [accuracy_score(y_train.iloc[va], pred_elo[va]) for _, va in cv.split(X_train)]
(pd.DataFrame(por_particion, index=[f"Partición {k}" for k in range(1, 6)]) * 100).round(2)

,Regresión logística,Random forest,XGBoost,Regla: mejor ranking,Regla: mayor Elo
Partición 1,69.14,69.13,69.31,65.98,67.14
Partición 2,71.03,71.06,70.96,67.87,69.38
Partición 3,70.42,70.61,70.48,67.61,69.24
Partición 4,69.54,69.57,69.53,66.72,68.35
Partición 5,67.26,67.37,67.17,63.76,65.87


### ¿Qué aprendió la logística?
Los coeficientes están sobre variables estandarizadas, así que se pueden comparar: un coeficiente más grande (en valor absoluto) pesa más.

In [11]:
logistica = finales["Regresión logística"].fit(X_train, y_train)
nombres = logistica.named_steps["prep"].get_feature_names_out()
coef = pd.Series(logistica.named_steps["modelo"].coef_[0], index=nombres)
coef.index = coef.index.str.replace("num__", "").str.replace("cat__", "")
coef.reindex(coef.abs().sort_values(ascending=False).index).head(15).round(3).to_frame("Coeficiente")

,Coeficiente
dif_elo_sup,0.673
dif_elo,0.362
dif_log_puntos,0.292
dif_edad,-0.232
dif_saque,0.185
dif_log_experiencia,0.161
dif_resto,0.109
dif_pct_sup,-0.106
dif_dias_desc,-0.095
ronda_F,0.093


### ¿Qué variables usa XGBoost?
La importancia por ganancia mide cuánto mejora la predicción cada vez que el modelo usa esa variable para dividir.

In [12]:
xgb = finales["XGBoost"].fit(X_train, y_train)
nombres = xgb.named_steps["prep"].get_feature_names_out()
imp = pd.Series(xgb.named_steps["modelo"].get_booster().get_score(importance_type="gain"))
imp.index = [nombres[int(i[1:])] for i in imp.index]
imp.index = imp.index.str.replace("num__", "").str.replace("cat__", "")
(imp / imp.sum() * 100).sort_values(ascending=False).head(15).round(1).to_frame("Importancia (%)")

,Importancia (%)
dif_elo,29.3
dif_elo_sup,23.0
dif_log_puntos,4.3
missingindicator_dif_ace,2.5
dif_log_rank,2.5
dif_dias_desc,2.3
dif_saque,1.9
best_of,1.9
missingindicator_dif_saque,1.7
nivel_G,1.6


### Entrenar con todo el periodo 2001–2022 y guardar
Los tres modelos se reentrenan con todo el entrenamiento y se guardan. En el paso 6 se evalúan en la prueba (2023–2026), que hasta ahora no se ha usado.

In [13]:
rf = finales["Random forest"].fit(X_train, y_train)
joblib.dump({"modelos": {"Regresión logística": logistica, "Random forest": rf, "XGBoost": xgb},
             "numericas": NUMERICAS, "categoricas": CATEGORICAS},
            os.path.join(CARPETA, "modelos.joblib"))
tabla_cv.to_csv(os.path.join(CARPETA, "resultados_cv.csv"), index=False)
print("Guardados: modelos.joblib y resultados_cv.csv")

Guardados: modelos.joblib y resultados_cv.csv


In [14]:
(test[test["anio"]== 2026]
 .groupby("torneo").agg(partidos=("match_id", "size"), desde=("fecha", "min"))
 .sort_values("desde"))

,partidos,desde
torneo,,
Brisbane,31,2026-01-04
United Cup,25,2026-01-05
Hong Kong,27,2026-01-05
Adelaide,26,2026-01-12
Auckland,27,2026-01-12
...,...,...
Madrid Masters,94,2026-04-22
Rome Masters,94,2026-05-06
Hamburg,31,2026-05-17


### Conclusión del paso 5
 
**Validación:** 5 particiones temporales dentro de 2001–2022 (cada una entrena con los años anteriores y valida con los siguientes), con el preprocesamiento dentro del Pipeline. Las reglas se evaluaron en los mismos bloques.
 
**Hiperparámetros elegidos** (por log loss):
- Regresión logística: C = 0,1 (regularización moderada).
- Random forest: profundidad máxima 14, mínimo 20 partidos por hoja.
- XGBoost: profundidad 3, tasa de aprendizaje 0,03 (árboles simples y pasos pequeños).
 
| Modelo | Exactitud | AUC | Log loss | Tiempo |
|---|---|---|---|---|
| Línea base (siempre A) | 49,74 % ± 0,41 | – | – | – |
| Regla: mejor ranking | 66,39 % ± 1,47 | – | – | – |
| Regla: mayor Elo | 68,00 % ± 1,33 | – | – | – |
| **Regresión logística** | **69,48 % ± 1,29** | **0,7679** | **0,5726** | **3 s** |
| Random forest | 69,55 % ± 1,29 | 0,7666 | 0,5752 | 31 s |
| XGBoost | 69,49 % ± 1,31 | 0,7677 | 0,5726 | 6 s |
 
- **Los tres modelos superan la regla del ranking en unos 3,1 puntos** y la regla del Elo en 1,5.
- **Entre ellos hay un empate:** la diferencia máxima es de 0,07 puntos, muy por debajo de la variación entre particiones (±1,3). La relación entre las variables y el resultado es casi lineal, así que los árboles no encuentran mucho más que la logística.
- **Candidata:** la regresión logística. Tiene el mejor AUC, empata en log loss con XGBoost, es 10 veces más rápida que el random forest y se puede interpretar. La elección final se hace en el paso 6, con los datos de prueba.
 
**Redundancia:** la logística con todas las variables (AUC 0,7678) supera a la versión sin los pares redundantes (0,7621). La regularización controla la redundancia, así que se conservan todas.
 
**¿Qué aprendió la logística?** Las variables que más pesan son el Elo por superficie (0,673), el Elo general (0,362) y los puntos (0,292). Además:
- **Edad (−0,232):** a igual nivel, gana el más joven. Por sí sola la edad casi no predecía (AUC 0,514): su efecto aparece al controlar por las demás variables.
- **Viene de la qualy (+0,075) o tiene invitación:** el ranking los subestima. El de la qualy llega con 2 o 3 partidos de ritmo; el invitado suele ser una promesa o un jugador local.
- **Signos "equivocados":** `dif_log_rank` sale positivo y `dif_pct_sup` negativo porque están muy correlacionados con los puntos (−0,91) y con el Elo por superficie (0,79). El modelo reparte el efecto entre variables parecidas: los coeficientes de variables correlacionadas no se interpretan uno por uno.
 
**¿Qué usa XGBoost?** El 52 % de la importancia está en el Elo (general 29,3 %, superficie 23,0 %). Aparecen también la marca de "faltan estadísticas de saque" (indica Copa Davis o jugadores con poca trayectoria) y `best_of`, que la logística no puede aprovechar sola: en partidos a 5 sets el favorito gana más.